# Polymer RL-Assisted MPC Horizon Selection With Dueling DDQN

This notebook keeps the current unified polymer horizon workflow, but swaps the legacy discrete agent for a standard dueling Double DQN implementation. The reward, state construction, horizon recipe mapping, plotting, and MPC comparison flow remain aligned with the existing unified horizon notebook.

In [ ]:
from pathlib import Path
import os

from systems.polymer.data_io import canonical_baseline_path
from utils.notebook_setup import prepare_polymer_notebook_env, print_notebook_summary

# User-editable run controls.
RUN_MODE = "nominal"  # "nominal" | "disturb"
STATE_MODE = "standard"  # "standard" | "mismatch"
STYLE_PROFILE = "hybrid"  # "hybrid" | "paper" | "debug"
SAVE_PDF = False

# Optional directory overrides. Leave as None to use Polymer/Data and Polymer/Results.
POLYMER_DATA_DIR_OVERRIDE = None
POLYMER_RESULTS_DIR_OVERRIDE = None

# Optional naming/path overrides. Leave as None to use the canonical mode-based defaults.
RESULT_PREFIX_OVERRIDE = None
COMPARE_PREFIX_OVERRIDE = None
BASELINE_MPC_PATH_OVERRIDE = None

# Optional run-size overrides. Leave as None to use the defaults from RUN_PROFILE_MAP.
N_TESTS_OVERRIDE = None
SET_POINTS_LEN_OVERRIDE = None
WARM_START_OVERRIDE = None
TEST_CYCLE_OVERRIDE = None
PLOT_START_EPISODE_OVERRIDE = None
COMPARE_START_EPISODE_OVERRIDE = None

REPO_ROOT, DATA_DIR, RESULT_DIR = prepare_polymer_notebook_env(
    data_dir_override=POLYMER_DATA_DIR_OVERRIDE,
    results_dir_override=POLYMER_RESULTS_DIR_OVERRIDE,
)
os.chdir(REPO_ROOT)

RUN_PROFILE_MAP = {
    "nominal": {
        "use_disturbance": False,
        "baseline_mpc_path": canonical_baseline_path(REPO_ROOT, "nominal", data_override=POLYMER_DATA_DIR_OVERRIDE),
        "result_prefix": "dueling_horizon_nominal_unified",
        "compare_prefix": "nominal_compare_dueling_horizon_unified",
        "plot_start_episode": 5,
        "compare_start_episode": 1,
        "compare_mode": "nominal",
    },
    "disturb": {
        "use_disturbance": True,
        "baseline_mpc_path": canonical_baseline_path(REPO_ROOT, "disturb", data_override=POLYMER_DATA_DIR_OVERRIDE),
        "result_prefix": "dueling_horizon_disturb_unified",
        "compare_prefix": "disturb_compare_dueling_horizon_unified",
        "plot_start_episode": 5,
        "compare_start_episode": 2,
        "compare_mode": "disturb",
    },
}
RUN_PROFILE = RUN_PROFILE_MAP[RUN_MODE]
RESULT_PREFIX = RESULT_PREFIX_OVERRIDE or RUN_PROFILE["result_prefix"]
COMPARE_PREFIX = COMPARE_PREFIX_OVERRIDE or RUN_PROFILE["compare_prefix"]
BASELINE_MPC_PATH = Path(BASELINE_MPC_PATH_OVERRIDE).expanduser() if BASELINE_MPC_PATH_OVERRIDE else RUN_PROFILE["baseline_mpc_path"]

print_notebook_summary(
    "Polymer dueling horizon configuration",
    {
        "Polymer data dir": DATA_DIR,
        "Polymer results": RESULT_DIR,
        "Run mode": RUN_MODE,
        "State mode": STATE_MODE,
        "Baseline MPC": BASELINE_MPC_PATH,
    },
)


In [ ]:
import random

import numpy as np
import torch

from DuelingDQN.dueling_dqn_agent import DuelingDQNAgent
from Simulation.system_functions import PolymerCSTR
from systems.polymer import (
    HORIZON_CONTROL_GRID,
    HORIZON_PREDICT_GRID,
    POLYMER_DELTA_T_HOURS,
    POLYMER_DESIGN_PARAMS,
    POLYMER_INPUT_BOUNDS,
    POLYMER_OBSERVER_POLES,
    POLYMER_RL_SETPOINTS_PHYS,
    POLYMER_SETPOINT_RANGE_PHYS,
    POLYMER_SS_INPUTS,
    POLYMER_SYSTEM_METADATA,
    POLYMER_SYSTEM_PARAMS,
    load_polymer_system_data,
)
from utils.helpers import apply_min_max, build_horizon_recipes
from utils.plotting import compare_mpc_rl_from_dirs, plot_horizon_results
from utils.rewards import make_reward_fn_relative_QR
from utils.state_features import get_rl_state_dim
from utils.horizon_runner_dueling import run_dueling_dqn_mpc_horizon_supervisor


In [ ]:
# Build the polymer plant and load the canonical polymer data bundle.
system_params = POLYMER_SYSTEM_PARAMS.copy()
system_design_params = POLYMER_DESIGN_PARAMS.copy()
system_steady_state_inputs = POLYMER_SS_INPUTS.copy()
delta_t = POLYMER_DELTA_T_HOURS

cstr = PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t)
steady_states = {"ss_inputs": cstr.ss_inputs, "y_ss": cstr.y_ss}

setpoint_y = POLYMER_SETPOINT_RANGE_PHYS.copy()
u_min = POLYMER_INPUT_BOUNDS["u_min"].copy()
u_max = POLYMER_INPUT_BOUNDS["u_max"].copy()

system_data = load_polymer_system_data(
    REPO_ROOT,
    steady_states=steady_states,
    setpoint_y=setpoint_y,
    u_min=u_min,
    u_max=u_max,
    n_inputs=2,
    data_override=POLYMER_DATA_DIR_OVERRIDE,
)

A_aug = system_data["A_aug"]
B_aug = system_data["B_aug"]
C_aug = system_data["C_aug"]
data_min = system_data["data_min"]
data_max = system_data["data_max"]
min_max_dict = system_data["min_max_dict"]

inputs_number = 2
y_sp_scenario = POLYMER_RL_SETPOINTS_PHYS.copy()
y_sp_scenario = apply_min_max(y_sp_scenario, data_min[inputs_number:], data_max[inputs_number:]) - apply_min_max(
    steady_states["y_ss"], data_min[inputs_number:], data_max[inputs_number:]
)


In [ ]:
# User-editable experiment defaults.
# These defaults are chosen for supervisory MPC decisions rather than high-frequency benchmark tasks.
DEFAULT_N_TESTS = 200
DEFAULT_SET_POINTS_LEN = 200
DEFAULT_WARM_START = 5
DEFAULT_TEST_CYCLE = [False, False, False, False, False]

n_tests = RUN_PROFILE.get("n_tests", DEFAULT_N_TESTS) if N_TESTS_OVERRIDE is None else int(N_TESTS_OVERRIDE)
set_points_len = RUN_PROFILE.get("set_points_len", DEFAULT_SET_POINTS_LEN) if SET_POINTS_LEN_OVERRIDE is None else int(SET_POINTS_LEN_OVERRIDE)
warm_start = RUN_PROFILE.get("warm_start", DEFAULT_WARM_START) if WARM_START_OVERRIDE is None else int(WARM_START_OVERRIDE)
TEST_CYCLE = list(RUN_PROFILE.get("test_cycle", DEFAULT_TEST_CYCLE)) if TEST_CYCLE_OVERRIDE is None else list(TEST_CYCLE_OVERRIDE)
PLOT_START_EPISODE = RUN_PROFILE.get("plot_start_episode", 1) if PLOT_START_EPISODE_OVERRIDE is None else int(PLOT_START_EPISODE_OVERRIDE)
COMPARE_START_EPISODE = RUN_PROFILE.get("compare_start_episode", PLOT_START_EPISODE) if COMPARE_START_EPISODE_OVERRIDE is None else int(COMPARE_START_EPISODE_OVERRIDE)

# Observer setup: the shared runner computes L internally from these poles.
poles = POLYMER_OBSERVER_POLES.copy()

# Reproducibility controls.
SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Horizon-action library.
PREDICT_GRID = list(range(8, 20))
CONTROL_GRID = list(range(3, 10))
HORIZON_RECIPES = build_horizon_recipes(PREDICT_GRID, CONTROL_GRID)
STATE_DIM = get_rl_state_dim(A_aug.shape[0], C_aug.shape[0], B_aug.shape[1], STATE_MODE)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Dueling DDQN defaults.
HIDDEN_LAYERS = [512, 512, 512]
BUFFER_SIZE = 40_000
DECISION_INTERVAL = 4
USE_SHIFTED_MPC_WARM_START = False  # False matches the legacy zero-initialized MPC solve.
TARGET_UPDATE = "hard"  # Hard target copies are the default for this discrete-control path.
HARD_UPDATE_INTERVAL = 2_000
TAU = 0.005
EPS_DECAY_MODE = "linear"
EPS_DECAY_STEPS = 15_000

predict_h = 9
cont_h = 3
Q1_penalty = 5.0
Q2_penalty = 1.0
R1_penalty = 1.0
R2_penalty = 1.0

k_rel = np.array([0.003, 0.0003])
band_floor_phys = np.array([0.006, 0.07])
Q_diag = np.array([518.0, 90.0])
R_diag = np.array([90.0, 90.0])
reward_params, reward_fn = make_reward_fn_relative_QR(
    data_min,
    data_max,
    n_inputs=2,
    k_rel=k_rel,
    band_floor_phys=band_floor_phys,
    Q_diag=Q_diag,
    R_diag=R_diag,
    tau_frac=0.7,
    gamma_out=0.5,
    gamma_in=0.5,
    beta=7.0,
    gate="geom",
    lam_in=1.0,
    bonus_kind="exp",
    bonus_k=12.0,
    bonus_p=0.6,
    bonus_c=20.0,
)

nominal_qs = 459.0
nominal_qi = 108.0
nominal_hA = 1.05e6
qi_change = 0.85
qs_change = 1.3
ha_change = 0.85

print(f"Using device: {DEVICE}")
print(f"State dimension: {STATE_DIM}")
print(f"Number of horizon actions: {len(HORIZON_RECIPES)}")
print(reward_params)

dueling_agent = DuelingDQNAgent(
    state_dim=STATE_DIM,
    action_dim=len(HORIZON_RECIPES),
    hidden_dim=HIDDEN_LAYERS,
    gamma=0.99,
    lr=1e-4,
    batch_size=128,
    buffer_size=BUFFER_SIZE,
    grad_clip_norm=10.0,
    double_dqn=True,
    target_update=TARGET_UPDATE,
    tau=TAU,
    hard_update_interval=HARD_UPDATE_INTERVAL,
    activation="relu",
    use_layer_norm=False,
    dropout=0.0,
    device=DEVICE,
    eps_start=0.30,
    eps_end=0.01,
    eps_decay_rate=0.99995,
    eps_decay_steps=EPS_DECAY_STEPS,
    eps_decay_mode=EPS_DECAY_MODE,
)

print_notebook_summary(
    "Resolved dueling horizon parameters",
    {
        "seed": SEED,
        "n_tests": n_tests,
        "set_points_len": set_points_len,
        "warm_start": warm_start,
        "plot_start_episode": PLOT_START_EPISODE,
        "compare_start_episode": COMPARE_START_EPISODE,
        "target_update": TARGET_UPDATE,
        "eps_decay_mode": EPS_DECAY_MODE,
    },
)


In [ ]:
dueling_cfg = {
    "mode": RUN_MODE,
    "state_mode": STATE_MODE,
    "predict_h": predict_h,
    "cont_h": cont_h,
    "decision_interval": DECISION_INTERVAL,
    "warm_start": warm_start,
    "test_cycle": TEST_CYCLE,
    "n_tests": n_tests,
    "set_points_len": set_points_len,
    "use_shifted_mpc_warm_start": USE_SHIFTED_MPC_WARM_START,
    "nominal_qi": nominal_qi,
    "nominal_qs": nominal_qs,
    "nominal_ha": nominal_hA,
    "qi_change": qi_change,
    "qs_change": qs_change,
    "ha_change": ha_change,
    "b_min": system_data["b_min"],
    "b_max": system_data["b_max"],
    "Q1_penalty": Q1_penalty,
    "Q2_penalty": Q2_penalty,
    "R1_penalty": R1_penalty,
    "R2_penalty": R2_penalty,
}

runtime_ctx = {
    "system": PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t),
    "y_sp_scenario": y_sp_scenario,
    "steady_states": steady_states,
    "min_max_dict": min_max_dict,
    "agent": dueling_agent,
    "A_aug": A_aug,
    "B_aug": B_aug,
    "C_aug": C_aug,
    "poles": poles,
    "data_min": data_min,
    "data_max": data_max,
    "horizon_recipes": HORIZON_RECIPES,
    "reward_fn": reward_fn,
    "system_metadata": POLYMER_SYSTEM_METADATA,
    "reward_params": reward_params,
}

result_bundle = run_dueling_dqn_mpc_horizon_supervisor(dueling_cfg=dueling_cfg, runtime_ctx=runtime_ctx)
result_bundle["mpc_path_or_dir"] = RUN_PROFILE["baseline_mpc_path"]
result_bundle["reward_params"] = reward_params
result_bundle["run_profile"] = RUN_PROFILE

print(f"nFE: {result_bundle['nFE']}")
print(f"Avg reward samples: {len(result_bundle['avg_rewards'])}")


In [ ]:
out_dir_rl = plot_horizon_results(
    result_bundle=result_bundle,
    plot_cfg={
        "directory": RESULT_DIR,
        "prefix_name": RESULT_PREFIX,
        "start_episode": PLOT_START_EPISODE,
        "recipe_counts": True,
        "save_pdf": SAVE_PDF,
        "style_profile": STYLE_PROFILE,
    },
)

out_dir_cmp = compare_mpc_rl_from_dirs(
    rl_dir=out_dir_rl,
    mpc_path_or_dir=BASELINE_MPC_PATH,
    reward_fn=reward_fn,
    directory=RESULT_DIR,
    prefix_name=COMPARE_PREFIX,
    compare_mode=RUN_PROFILE["compare_mode"],
    start_episode=COMPARE_START_EPISODE,
    save_pdf=SAVE_PDF,
    style_profile=STYLE_PROFILE,
)

print(f"RL plots saved to      : {out_dir_rl}")
print(f"Comparison plots saved : {out_dir_cmp}")
